# Pipeline con dos modelos para predecir

In [1]:
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

# ---------- Cargar modelo y datos normalizados
model_1 = load_model("../../models/model_7_6/model_7_6_dayOfYear.keras")
model_2 = load_model("../../models/model_7_9/model_7_9_dayOfYear.keras")

with open("../../data/normalized/df_normalized.pk1", "rb") as f:
    df_data = pickle.load(f)


In [2]:
import pickle
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_rows", None)   # Muestra todas las filas
pd.set_option("display.max_columns", None)  # Muestra todas las columnas
pd.set_option("display.width", None)     # No corta la tabla en varias líneas
pd.set_option("display.max_colwidth", None)  # Muestra el contenido de celdas completo

In [3]:
with open("../../data/normalized/df_normalized_dayOfYear.pk1", "rb") as f:
    df_data = pickle.load(f)

In [4]:
df_model = df_data.drop(columns=[
    "ride_id",
    "ended_at",
    "time_hms_ms",
    "member_casual",
    "start_station_id",
    "end_station_id",
    "month",
    "day",
    "temperature",
    "wind_speed",
    "precipitation",
    "relative_humidity",
    "snow_depth",
    "hour_float",
    "rideable_type_classic_bike",
    "rideable_type_electric_bike",
    "rideable_type_docked_bike",
    "member_casual",
    "member_casual_bool",
    "duration_min",
    "dayofyear"
    #"started_at"
])

# Se cambia la precision por minutos

In [5]:
# Se redonde al minuto mas cercano
df_model["started_minute"] = df_model["started_at"].dt.round("min")

In [6]:
df_agg = df_model.groupby([
    "start_station_idx",
    "end_station_idx",
    "started_minute"
]).agg(
    #n_viajes=("ride_id", "count"),
    year=("year", "first"),
    temp_std=("temp_std", "first"),
    wind_std=("wind_std", "first"),
    rel_humidity_std=("rel_humidity_std", "first"),
    precipitation_std=("precipitation_std", "first"),
    snow_depth_std=("snow_depth_std", "first"),
    hour_sin=("hour_sin", "first"),
    hour_cos=("hour_cos", "first"),
    month_sin=("month_sin", "first"),
    month_cos=("month_cos", "first"),
    event=("event", "any"),  # True si al menos un dato es true
    normal_day=("day_type_Normal", "any"),  # True si al menos un dato es true
    weekend_day=("day_type_Weekend", "any"),  # True si al menos un dato es true
    holiday_day=("day_type_Holiday", "any"),  # True si al menos un dato es true
    doy_sin=("doy_sin", "first"),
    doy_cos=("doy_cos", "first"),
    #member_casual=("member_casual_bool", "any"),  # True si al menos un dato es true
    #classic_bike=("rideable_type_classic_bike", "any"),  # True si al menos un dato es true
    #docked_bike=("rideable_type_docked_bike", "any"),  # True si al menos un dato es true
    #electric_bike=("rideable_type_electric_bike", "any"),  # True si al menos un dato es true
    #duration_min_mean=("duration_min", "mean"),
).reset_index()

In [7]:
df_agg.shape

(9073410, 19)

# Se obtiene los datos

In [8]:
x_context = df_agg.drop(columns=[
    "start_station_idx",
    "end_station_idx",
    "started_minute"
])

x_start = df_agg[[
    "start_station_idx",
]]

x_end = df_agg[[
    "end_station_idx",
]]

#y_real = df_agg["n_viajes_agg"]

In [9]:
x_context.shape

(9073410, 16)

In [10]:
x_context.dtypes

year                   int64
temp_std             float64
wind_std             float64
rel_humidity_std     float64
precipitation_std    float64
snow_depth_std       float64
hour_sin             float64
hour_cos             float64
month_sin            float64
month_cos            float64
event                   bool
normal_day              bool
weekend_day             bool
holiday_day             bool
doy_sin              float64
doy_cos              float64
dtype: object

# Se realiza la primera predicción

In [11]:
# Predicciones como probabilidades
y_predictions_prob = model_1.predict({
    'start_station': x_start,
    'end_station': x_end,
    'context': x_context
}, batch_size=256)

print("Shape de predicciones:", y_predictions_prob.shape)

35444/35444 ━━━━━━━━━━━━━━━━━━━━ 33s 937us/step
Shape de predicciones: (9073410, 8)


In [12]:
# La clase 1 (1 viaje) se compone de la clase 0 a la 3

max_prob_class_1 = np.max(y_predictions_prob[:, 0:4], axis=1)

In [ ]:
target = 8671457  # número de viajes reales clase 1 en los datos disponibles

# Ordenas de mayor a menor
sorted_probs = np.sort(max_prob_class_1)[::-1]

# El umbral es el valor en la posición target - 1
threshold = sorted_probs[target - 1]

print("Umbral:", threshold)

Umbral: 0.23296134


In [14]:
y_predictions_class_1 = np.where(max_prob_class_1 >= threshold, True, False)

# Contabilizar los 1 en y_predictions_1
count_1 = np.sum(y_predictions_class_1 == True)
count_1

np.int64(8671457)

# Se realiza la segunda predicción

In [15]:
df_agg['predict_class_1'] = y_predictions_class_1

In [16]:
df_agg.head()

,start_station_idx,end_station_idx,started_minute,year,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_sin,hour_cos,month_sin,month_cos,event,normal_day,weekend_day,holiday_day,doy_sin,doy_cos,predict_class_1
0,0,0,2022-02-11 13:38:00,2022,-1.379839,1.021427,1.326397,-0.099704,-0.056148,-0.416215,-0.909266,0.866025,5.000000e-01,False,False,False,True,0.661635,0.749826,True
1,0,0,2022-03-05 15:03:00,2022,-0.612834,-0.276355,-0.807078,-0.099704,-0.056148,-0.717569,-0.696487,1.000000,6.123234e-17,False,False,True,False,0.891981,0.452072,True
2,0,0,2022-03-06 10:55:00,2022,-1.156238,4.113864,0.687365,-0.099704,-0.056148,0.281155,-0.959662,1.000000,6.123234e-17,False,False,True,False,0.899631,0.436651,True
3,0,0,2022-03-20 17:06:00,2022,-0.424108,-0.883141,-1.019270,-0.099704,-0.056148,-0.972183,-0.234223,1.000000,6.123234e-17,False,False,True,False,0.977848,0.209315,True
4,0,0,2022-03-20 17:11:00,2022,-0.410327,-0.825775,-1.038403,-0.099704,-0.056148,-0.976781,-0.214238,1.000000,6.123234e-17,False,False,True,False,0.977848,0.209315,True


In [17]:
df_agg = df_agg[df_agg['predict_class_1'] == False]

In [18]:
df_agg.head()

,start_station_idx,end_station_idx,started_minute,year,temp_std,wind_std,rel_humidity_std,precipitation_std,snow_depth_std,hour_sin,hour_cos,month_sin,month_cos,event,normal_day,weekend_day,holiday_day,doy_sin,doy_cos,predict_class_1
73854,100,1399,2022-05-24 18:13:00,2022,0.170553,1.231771,-1.007791,-0.099704,-0.056148,-0.998270,0.058798,5.000000e-01,-8.660254e-01,False,True,False,False,0.615285,-0.788305,False
73936,105,1393,2023-06-24 22:52:00,2023,1.258757,0.849328,-0.916910,-0.099704,-0.056148,-0.292302,0.956326,1.224647e-16,-1.000000e+00,False,False,True,False,0.128748,-0.991677,False
74039,107,1303,2022-06-25 20:26:00,2022,0.835672,1.868157,0.040681,-0.099704,-0.056148,-0.804289,0.594238,1.224647e-16,-1.000000e+00,False,False,True,False,0.111659,-0.993747,False
74041,107,1617,2023-09-23 19:19:00,2023,0.736619,0.303198,-0.397458,-0.099704,-0.056148,-0.941299,0.337574,-1.000000e+00,-1.836970e-16,False,False,True,False,-0.991114,-0.133015,False
75441,144,1343,2022-07-08 16:30:00,2022,0.976758,0.828467,0.449772,-0.099704,-0.056148,-0.923963,-0.382482,-5.000000e-01,-8.660254e-01,True,True,False,False,-0.111659,-0.993747,False


In [19]:
df_agg.shape

(401953, 20)

# Se obtienen los datos

In [20]:
x_context = df_agg.drop(columns=[
    "start_station_idx",
    "end_station_idx",
    "started_minute",
    "predict_class_1"
])

x_start = df_agg[[
    "start_station_idx",
]]

x_end = df_agg[[
    "end_station_idx",
]]

In [21]:
x_context.shape

(401953, 16)

In [22]:
x_context.dtypes

year                   int64
temp_std             float64
wind_std             float64
rel_humidity_std     float64
precipitation_std    float64
snow_depth_std       float64
hour_sin             float64
hour_cos             float64
month_sin            float64
month_cos            float64
event                   bool
normal_day              bool
weekend_day             bool
holiday_day             bool
doy_sin              float64
doy_cos              float64
dtype: object

In [23]:
y_pred = model_2.predict({
    'start_station': x_start,
    'end_station': x_end,
    'context': x_context
})

12562/12562 ━━━━━━━━━━━━━━━━━━━━ 9s 672us/step


In [24]:
y_pred

array([[2.3073592],
       [2.237206 ],
       [2.4782233],
       ...,
       [1.8759593],
       [1.8673116],
       [1.9218893]], shape=(401953, 1), dtype=float32)

In [25]:
y_pred_int = np.rint(y_pred).astype(int)  # redondear al entero más cercano
y_pred_int

array([[2],
       [2],
       [2],
       ...,
       [2],
       [2],
       [2]], shape=(401953, 1))

Se muestran cuantos datos hay de cada tipo

In [26]:
import numpy as np

unique, counts = np.unique(y_pred_int, return_counts=True)
print(unique)
print(counts)


[2 3 4 5]
[374320  26489   1139      5]


Los datos correctos son

2 -> 372173

3 -> 24990

4 ->  4098

5 -> 692